# 01 — Dataset Download
**Project:** Real-Time Audio Fall Detection (TinyML)

This notebook downloads all raw datasets to Google Drive. Each cell is **idempotent** — 
re-running a cell will skip datasets that are already present.

### Prerequisites
- A Google account with sufficient Drive storage (~15-20 GB recommended).
- A Kaggle account with an API token (`KAGGLE_API_TOKEN`).

### Datasets downloaded
| # | Dataset | Source | Role |
|---|---------|--------|------|
| 1 | DESED / DCASE | Zenodo | Domestic noise (water, vacuum) |
| 2 | CAUCAFall | Mendeley | Human fall impact sounds |
| 3 | Water Leakage | GitHub | Bathroom faucet/toilet sounds |
| 4 | Human Screaming | Kaggle | Vocal distress sounds |
| 5 | FSD50K | Zenodo | Generic sounds (splashing, sink) |
| 6 | SAFE | Kaggle | Additional fall simulations |
| 7 | AudioSet (filtered) | Google | Shower / water variety |

## 1 · Setup

In [2]:
from google.colab import drive
import os, shutil, time, subprocess, zipfile, tarfile, requests
from pathlib import Path

# ── Mount Google Drive ──
drive.mount("/content/drive")

# ── Project paths ──
BASE_DIR = "/content/drive/MyDrive/DL-Project"
RAW_DIR  = os.path.join(BASE_DIR, "raw")

DATASETS = ["DESED", "CAUCAFall", "WaterLeakage", "HumanScreaming", "FSD50K", "SAFE", "AudioSet"]

for ds in DATASETS:
    os.makedirs(os.path.join(RAW_DIR, ds), exist_ok=True)

print(f"Base directory : {BASE_DIR}")
print(f"Raw data directory : {RAW_DIR}")
print()
for ds in DATASETS:
    print(f"  - {ds}/")

MessageError: [dfs_ephemeral] Credentials propagation unsuccessful

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import getpass, os

if "KAGGLE_API_TOKEN" not in os.environ:
    os.environ["KAGGLE_API_TOKEN"] = getpass.getpass("Enter your KAGGLE_API_TOKEN: ")

# Install Kaggle CLI (quiet)
subprocess.run(["pip", "install", "-q", "kaggle"], check=True)

# Verify authentication
result = subprocess.run(["kaggle", "datasets", "list", "--max-size", "1"],
                        capture_output=True, text=True)
if result.returncode == 0:
    print("Kaggle CLI authenticated successfully.")
else:
    print("Kaggle authentication FAILED — check your token.")
    print(result.stderr)

In [ ]:
# ── Helper Utilities ───────────────────────────────────────────

def download_with_wget(url, dest_path, desc=""):
    """Download a file using wget."""
    label = desc or os.path.basename(dest_path)
    print(f"  Downloading {label} ...")
    start = time.time()
    result = subprocess.run(["wget", "-q", "--show-progress", "-O", str(dest_path), url])
    elapsed = time.time() - start
    if result.returncode == 0 and os.path.exists(dest_path):
        size_mb = os.path.getsize(dest_path) / (1024 * 1024)
        print(f"  -> Done ({size_mb:.1f} MB, {elapsed:.0f}s)")
        return True
    else:
        print(f"  -> Download FAILED (exit code {result.returncode})")
        return False


def extract_archive(archive_path, dest_dir):
    """Extract zip / tar.gz / tar and remove the archive afterwards."""
    name = os.path.basename(archive_path)
    print(f"  Extracting {name} ...")
    try:
        if archive_path.endswith(".zip"):
            with zipfile.ZipFile(archive_path, "r") as zf:
                zf.extractall(dest_dir)
        elif archive_path.endswith((".tar.gz", ".tgz")):
            with tarfile.open(archive_path, "r:gz") as tf:
                tf.extractall(dest_dir)
        elif archive_path.endswith(".tar"):
            with tarfile.open(archive_path, "r:") as tf:
                tf.extractall(dest_dir)
        else:
            print(f"  -> Unknown archive format: {name}")
            return
        os.remove(archive_path)
        print(f"  -> Extracted to {dest_dir}")
    except Exception as e:
        print(f"  -> Extraction failed: {e}")


def count_files(directory):
    """Recursively count files in a directory."""
    return sum(1 for _ in Path(directory).rglob("*") if _.is_file())


def dir_size_mb(directory):
    """Total size of a directory in MB."""
    return sum(f.stat().st_size for f in Path(directory).rglob("*") if f.is_file()) / (1024 * 1024)


def is_populated(directory):
    """True if directory already contains files (for idempotent re-runs)."""
    return count_files(directory) > 0


print("Helper utilities loaded.")

## 2 · Downloads
Run each cell below to download a dataset. Cells can be run independently and in any order.

In [ ]:
# ── 1/6  DESED / DCASE Domestic Sounds ────────────────────────
# Source : https://zenodo.org/records/4560938
# Role   : Domestic noise — water, vacuum, etc.

dataset_dir = os.path.join(RAW_DIR, "DESED")

if is_populated(dataset_dir):
    print(f"Skipping DESED — already has {count_files(dataset_dir)} files ({dir_size_mb(dataset_dir):.1f} MB)")
else:
    print("Downloading DESED / DCASE Domestic Sounds (Zenodo) ...")

    r = requests.get("https://zenodo.org/api/records/4560938")
    r.raise_for_status()
    files = r.json()["files"]

    total_mb = sum(f["size"] for f in files) / (1024 ** 2)
    print(f"  Found {len(files)} files ({total_mb:.0f} MB total):")
    for f in files:
        print(f"    - {f['key']}  ({f['size'] / (1024**2):.0f} MB)")
    print()

    for f in files:
        fname = f["key"]
        url   = f["links"]["self"]
        dest  = os.path.join(dataset_dir, fname)

        if os.path.exists(dest):
            print(f"  {fname} already exists, skipping")
            continue

        ok = download_with_wget(url, dest, desc=fname)

        if ok and fname.endswith((".zip", ".tar.gz", ".tgz")):
            extract_archive(dest, dataset_dir)

    print(f"\nDESED complete — {count_files(dataset_dir)} files ({dir_size_mb(dataset_dir):.1f} MB)")

In [ ]:
# ── 2/6  CAUCAFall Dataset ─────────────────────────────────────
# Source : https://data.mendeley.com/datasets/7w7fccy7ky/4
# Role   : High-fidelity human fall "thuds" and impact sounds.

dataset_dir = os.path.join(RAW_DIR, "CAUCAFall")

if is_populated(dataset_dir):
    print(f"Skipping CAUCAFall — already has {count_files(dataset_dir)} files ({dir_size_mb(dataset_dir):.1f} MB)")
else:
    print("Downloading CAUCAFall (Mendeley) ...")

    zip_path = os.path.join(dataset_dir, "caucafall.zip")

    # Mendeley S3 cache URL (standard pattern for public datasets)
    url = "https://prod-dcd-datasets-cache-zipfiles.s3.eu-west-1.amazonaws.com/7w7fccy7ky-4.zip"
    ok = download_with_wget(url, zip_path, desc="CAUCAFall")

    if ok and os.path.exists(zip_path) and os.path.getsize(zip_path) > 1024:
        extract_archive(zip_path, dataset_dir)
        print(f"CAUCAFall complete — {count_files(dataset_dir)} files ({dir_size_mb(dataset_dir):.1f} MB)")
    else:
        # Clean up partial/empty file
        if os.path.exists(zip_path):
            os.remove(zip_path)
        print("Automatic download failed.")
        print("  -> Manual fallback: go to https://data.mendeley.com/datasets/7w7fccy7ky/4")
        print(f"  -> Click 'Download All' and save the ZIP into: {dataset_dir}")

In [ ]:
# ── 3/6  Water Leakage Voice Data ──────────────────────────────
# Source : https://github.com/aliEmreOzturk/water_leakage_voice_data
# Role   : Bathroom faucet / toilet sounds for background filtering.

dataset_dir = os.path.join(RAW_DIR, "WaterLeakage")

if is_populated(dataset_dir):
    print(f"Skipping WaterLeakage — already has {count_files(dataset_dir)} files ({dir_size_mb(dataset_dir):.1f} MB)")
else:
    print("Cloning Water Leakage Voice Data (GitHub) ...")

    tmp_dir = os.path.join(RAW_DIR, "_tmp_water_clone")
    if os.path.exists(tmp_dir):
        shutil.rmtree(tmp_dir)

    subprocess.run([
        "git", "clone", "--depth", "1",
        "https://github.com/aliEmreOzturk/water_leakage_voice_data.git",
        tmp_dir
    ], check=True)

    # Move contents (skip .git) into the dataset folder
    for item in os.listdir(tmp_dir):
        if item == ".git":
            continue
        shutil.move(os.path.join(tmp_dir, item), os.path.join(dataset_dir, item))
    shutil.rmtree(tmp_dir)

    print(f"WaterLeakage complete — {count_files(dataset_dir)} files ({dir_size_mb(dataset_dir):.1f} MB)")

In [ ]:
# ── 4/6  Human Screaming Detection ─────────────────────────────
# Source : https://www.kaggle.com/datasets/whats2000/human-screaming-detection-dataset
# Role   : Vocal distress component of a fall.

dataset_dir = os.path.join(RAW_DIR, "HumanScreaming")

if is_populated(dataset_dir):
    print(f"Skipping HumanScreaming — already has {count_files(dataset_dir)} files ({dir_size_mb(dataset_dir):.1f} MB)")
else:
    print("Downloading Human Screaming Detection Dataset (Kaggle) ...")

    result = subprocess.run([
        "kaggle", "datasets", "download",
        "whats2000/human-screaming-detection-dataset",
        "-p", dataset_dir, "--unzip"
    ], capture_output=True, text=True)

    if result.returncode == 0:
        print(f"HumanScreaming complete — {count_files(dataset_dir)} files ({dir_size_mb(dataset_dir):.1f} MB)")
    else:
        print("Download FAILED:")
        print(result.stderr)

In [ ]:
# ── 5/6  FSD50K ────────────────────────────────────────────────
# Source : https://zenodo.org/records/4060432
# Role   : Large-scale generic sounds — "Splashing", "Sink" variety.
# NOTE   : This is a large dataset. Download may take a while.

dataset_dir = os.path.join(RAW_DIR, "FSD50K")

if is_populated(dataset_dir):
    print(f"Skipping FSD50K — already has {count_files(dataset_dir)} files ({dir_size_mb(dataset_dir):.1f} MB)")
else:
    print("Downloading FSD50K (Zenodo) ...")
    print("  NOTE: This dataset is large — expect a lengthy download.\n")

    r = requests.get("https://zenodo.org/api/records/4060432")
    r.raise_for_status()
    files = r.json()["files"]

    total_gb = sum(f["size"] for f in files) / (1024 ** 3)
    print(f"  Found {len(files)} files ({total_gb:.1f} GB total):")
    for f in files:
        print(f"    - {f['key']}  ({f['size'] / (1024**2):.0f} MB)")
    print()

    for f in files:
        fname = f["key"]
        url   = f["links"]["self"]
        dest  = os.path.join(dataset_dir, fname)

        if os.path.exists(dest):
            print(f"  {fname} already exists, skipping")
            continue

        ok = download_with_wget(url, dest, desc=fname)

        if ok and fname.endswith((".zip", ".tar.gz", ".tgz")):
            extract_archive(dest, dataset_dir)

    print(f"\nFSD50K complete — {count_files(dataset_dir)} files ({dir_size_mb(dataset_dir):.1f} MB)")

In [ ]:
# ── 6/6  SAFE — Sound Analysis for Fall Events ────────────────
# Source : https://www.kaggle.com/datasets/antonygarciag/fall-audio-detection-dataset
# Role   : Additional fall simulations for "Thud" variety.

dataset_dir = os.path.join(RAW_DIR, "SAFE")

if is_populated(dataset_dir):
    print(f"Skipping SAFE — already has {count_files(dataset_dir)} files ({dir_size_mb(dataset_dir):.1f} MB)")
else:
    print("Downloading SAFE (Kaggle) ...")

    result = subprocess.run([
        "kaggle", "datasets", "download",
        "antonygarciag/fall-audio-detection-dataset",
        "-p", dataset_dir, "--unzip"
    ], capture_output=True, text=True)

    if result.returncode == 0:
        print(f"SAFE complete — {count_files(dataset_dir)} files ({dir_size_mb(dataset_dir):.1f} MB)")
    else:
        print("Download FAILED:")
        print(result.stderr)

In [ ]:
# ── 7/7  AudioSet (Filtered — Shower / Water segments) ────────
# Source : https://research.google.com/audioset/
# Role   : Additional "Shower" and water variety from YouTube clips.
#
# AudioSet is YouTube-based: we download the segment CSVs, filter
# for relevant sound classes, then use yt-dlp to extract audio.

import csv, io

dataset_dir = os.path.join(RAW_DIR, "AudioSet")

if is_populated(dataset_dir):
    print(f"Skipping AudioSet — already has {count_files(dataset_dir)} files ({dir_size_mb(dataset_dir):.1f} MB)")
else:
    print("Setting up AudioSet filtered download ...")

    # Install yt-dlp (YouTube audio extraction)
    subprocess.run(["pip", "install", "-q", "yt-dlp"], check=True)

    # ── Target AudioSet class labels (mid codes) ──
    # Full ontology: https://research.google.com/audioset/ontology/index.html
    TARGET_LABELS = {
        "/m/09t49":   "Shower",
        "/m/07rjzl8": "Bathtub (filling or washing)",
        "/m/07pbtc8": "Water tap, faucet",
        "/m/05cj5r":  "Sink (filling or washing)",
        "/t/dd00038": "Rain on surface",
        "/m/07r04":   "Splash, splashing",
        "/m/0838f":   "Water",
        "/m/0j6m2":   "Waterfall",
    }

    print(f"  Target classes ({len(TARGET_LABELS)}):")
    for mid, name in TARGET_LABELS.items():
        print(f"    {mid} -> {name}")

    # ── Download segment CSVs ──
    SEGMENT_CSVS = {
        "balanced_train": "http://storage.googleapis.com/us_audioset/youtube_corpus/v1/csv/balanced_train_segments.csv",
        "eval":           "http://storage.googleapis.com/us_audioset/youtube_corpus/v1/csv/eval_segments.csv",
    }

    target_mids = set(TARGET_LABELS.keys())
    matched_segments = []

    for split_name, url in SEGMENT_CSVS.items():
        print(f"\n  Fetching {split_name} segment list ...")
        resp = requests.get(url)
        resp.raise_for_status()

        # AudioSet CSVs have 3 comment lines starting with #
        lines = [l for l in resp.text.strip().split("\n") if not l.startswith("#")]
        reader = csv.reader(lines, skipinitialspace=True)

        for row in reader:
            if len(row) < 4:
                continue
            ytid, start, end = row[0].strip(), row[1].strip(), row[2].strip()
            labels = set(l.strip().strip('"') for l in row[3:])

            if labels & target_mids:
                matched_labels = labels & target_mids
                label_names = [TARGET_LABELS[m] for m in matched_labels]
                matched_segments.append((ytid, float(start), float(end), label_names))

    print(f"\n  Found {len(matched_segments)} matching segments across all splits.")

    if len(matched_segments) == 0:
        print("  WARNING: No segments matched. Check label MIDs.")
    else:
        # ── Download audio segments using yt-dlp ──
        MAX_DOWNLOADS = 500  # Safety cap — adjust as needed
        print(f"  Downloading up to {MAX_DOWNLOADS} segments ...")

        success = 0
        fail    = 0
        start_time = time.time()

        for i, (ytid, t_start, t_end, label_names) in enumerate(matched_segments[:MAX_DOWNLOADS]):
            tag = "_".join(label_names).replace(" ", "-").replace("/", "-")[:40]
            out_file = os.path.join(dataset_dir, f"{ytid}_{t_start:.0f}_{t_end:.0f}_{tag}.wav")

            if os.path.exists(out_file):
                success += 1
                continue

            yt_url = f"https://www.youtube.com/watch?v={ytid}"
            cmd = [
                "yt-dlp",
                "--no-warnings", "-q",
                "--extract-audio", "--audio-format", "wav",
                "--download-sections", f"*{t_start:.1f}-{t_end:.1f}",
                "--output", out_file,
                "--no-playlist",
                "--socket-timeout", "15",
                yt_url,
            ]

            try:
                result = subprocess.run(cmd, capture_output=True, text=True, timeout=60)
                if result.returncode == 0 and os.path.exists(out_file):
                    success += 1
                else:
                    fail += 1
            except subprocess.TimeoutExpired:
                fail += 1

            # Progress update every 50 clips
            if (i + 1) % 50 == 0:
                elapsed = time.time() - start_time
                print(f"    [{i+1}/{min(len(matched_segments), MAX_DOWNLOADS)}]  "
                      f"OK: {success}  Failed: {fail}  ({elapsed:.0f}s)")

        elapsed = time.time() - start_time
        print(f"\n  AudioSet complete — {success} downloaded, {fail} failed ({elapsed:.0f}s)")
        print(f"  {count_files(dataset_dir)} files ({dir_size_mb(dataset_dir):.1f} MB)")

## 3 · Summary

In [ ]:
# ── Download Summary ───────────────────────────────────────────

print("=" * 60)
print("  DATASET DOWNLOAD SUMMARY")
print("=" * 60)

total_files = 0
total_size  = 0.0

for ds in DATASETS:
    ds_path  = os.path.join(RAW_DIR, ds)
    n_files  = count_files(ds_path)
    size     = dir_size_mb(ds_path)
    status   = "OK" if n_files > 0 else "MISSING"
    total_files += n_files
    total_size  += size
    print(f"  [{status:>7s}]  {ds:20s}  {n_files:6d} files  {size:8.1f} MB")

print("-" * 60)
print(f"  {'':9s}  {'TOTAL':20s}  {total_files:6d} files  {total_size:8.1f} MB")
print("=" * 60)

if total_files > 0:
    print(f"\n  All data stored at: {RAW_DIR}")
    print("  Next step: Run the data inventory / preprocessing notebook.")